In [8]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from sklearn.metrics import r2_score

In [9]:
df = pd.read_csv("Walmart.csv")

df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [10]:
df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True
)

In [11]:
df["Date"] = pd.to_datetime(df["Date"])

df["Year"] = df["Date"].dt.year

In [12]:
X = df.drop(
    ["Weekly_Sales", "Date"],
    axis=1
)

y = df["Weekly_Sales"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

In [14]:
numeric_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=0
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=0
    ),
    "SVM": SVR()
}

In [16]:
best_model = None
best_score = -999
best_name = ""

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)

    score = r2_score(
        y_test,
        pred
    )

    print(name, ":", score)

    if score > best_score:
        best_score = score
        best_model = pipe
        best_name = name

print("\nBest Model :", best_name)
print("Best Score :", best_score)

Linear Regression : 0.16443293821568783
Decision Tree : 0.8794964845391111
Random Forest : 0.9347335780238892
SVM : -0.026190964964191155

Best Model : Random Forest
Best Score : 0.9347335780238892


In [17]:
pickle.dump(
    best_model,
    open(
        "walmart_sales_model.pkl",
        "wb"
    )
)

print("Model Saved Successfully")

Model Saved Successfully
